In [1]:
import django

django.setup()

In [2]:
import logging
import sys

import pokebase as pb
from pokebase import APIMetadata, APIResource
from requests.exceptions import HTTPError

from typing import Any, Optional

from django.db import transaction
from django.db.models import Model

from pokedex.models import (
    Move,
    Pokemon,
    PokemonAbility,
    PokemonForm,
    PokemonFormType,
    PokemonMove,
    PokemonMoveVersion,
    PokemonSpecies,
    PokemonSpeciesDexEntry,
    PokemonSpeciesVariety,
    PokemonStat,
    PokemonType,
    VersionGameIndex,
)

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
    stream=sys.stdout,
    force=True,
)


logger = logging.getLogger(__name__)


class PokemonBuilder:
    def __get_api_metadata_as_dict(self, api_metadata: APIMetadata) -> dict[str, Any]:
        _ALLOWED_TYPES = (bool, str, int, float, APIMetadata, APIResource)

        if not isinstance(api_metadata, APIMetadata):
            raise Exception(f"objeto informado deve implementar {APIMetadata}")

        result = {}

        for key, value in vars(api_metadata).items():
            if value and not isinstance(value, _ALLOWED_TYPES):
                logger.warning(
                    f"[{key}] value deve ser {_ALLOWED_TYPES} e recebeu {type(value)}"
                )
                continue

            if type(value) == APIMetadata:
                result[key] = self.__get_api_metadata_as_dict(value)
                continue

            result[key] = value

        return result

    def __get_object_or_none(self, m: Model, **lkp: dict[str, Any]) -> Optional[Model]:
        try:
            return m.objects.get(**lkp)
        except m.DoesNotExist:
            return None

    def __get_pokemon(
        self, r: Optional[APIResource] = None
    ) -> tuple[Optional[Pokemon], bool]:
        resource = r if r and isinstance(r, APIResource) else self.resource

        return Pokemon.objects.get_or_create(
            name=resource.name,
            defaults={
                "name": resource.name,
                "base_experience": resource.base_experience,
                "height": resource.height,
                "is_default": resource.is_default,
                "order": resource.order,
                "weight": resource.weight,
                "sprites": self.__get_api_metadata_as_dict(resource.sprites),
                "cries": self.__get_api_metadata_as_dict(resource.cries),
            },
        )

    def __get_pokemonform(
        self, r: Optional[APIResource] = None
    ) -> tuple[Optional[PokemonForm], bool]:
        resource = r if r and isinstance(r, APIResource) else self.resource

        builder = PokemonBuilder(resource=str(resource.pokemon))
        pokemon = builder.build_pokemon(link_species=True)[0]

        return PokemonForm.objects.get_or_create(
            name=resource.name,
            defaults={
                "order": resource.order,
                "form_order": resource.form_order,
                "is_default": resource.is_default,
                "is_battle_only": resource.is_battle_only,
                "is_mega": resource.is_mega,
                "form_name": resource.form_name,
                "sprites": self.__get_api_metadata_as_dict(resource.sprites),
                "version_group": resource.version_group,
                "pokemon": pokemon,
            },
        )

    def __get_evolves_from_species(self) -> Optional[PokemonSpecies]:
        if not self.resource.species.evolves_from_species:
            return None

        return self.__get_object_or_none(
            PokemonSpecies, name=self.resource.species.evolves_from_species.name
        )

    def __link_abilities(self, pokemon: Pokemon):
        for a in self.resource.abilities:
            logger.debug("linking ability: %s" % a.ability)
            pokemon.abilities.add(
                PokemonAbility.objects.get_or_create(
                    slot=a.slot,
                    is_hidden=a.is_hidden,
                    ability=str(a.ability),
                )[0]
            )

    def __link_forms(self, pokemon: Pokemon):
        for f in self.resource.forms:
            logger.debug("linking form: %s" % f.name)
            form, _ = self.__get_pokemonform(f)

            logger.debug("linking pokemon form types...")
            self.__link_pokemon_form_types(form, f.types)

    def __link_game_indices(self, pokemon: Pokemon):
        for i in self.resource.game_indices:
            logger.debug(
                "linking index %s#%d a %s"
                % (i.version.name, i.game_index, pokemon.name)
            )
            pokemon.game_indices.add(
                VersionGameIndex.objects.get_or_create(
                    game_index=i.game_index, version=i.version.name
                )[0]
            )

    def __link_moves(self, pokemon: Pokemon):
        p_moves = []

        for m in self.resource.moves:
            logger.debug("linking move: %s" % str(m.move))
            r_move = pb.move(str(m.move))
            p_move_versions = []

            move, _ = Move.objects.get_or_create(
                name=r_move.name,
                defaults={
                    "accuracy": r_move.accuracy,
                    "effect_chance": r_move.effect_chance,
                    "pp": r_move.pp,
                    "priority": r_move.priority,
                    "power": r_move.power,
                    "damage_class": r_move.damage_class,
                    "generation": r_move.generation,
                    "target": r_move.target,
                    "type": r_move.type,
                    "pokemon": pokemon,
                },
            )

            p_move, _ = PokemonMove.objects.get_or_create(move=move)

            for v in m.version_group_details:
                p_move_versions.append(
                    PokemonMoveVersion.objects.get_or_create(
                        move_learn_method=v.move_learn_method,
                        level_learned_at=v.level_learned_at,
                        version_group=v.version_group,
                        order=v.order,
                    )[0]
                )

            p_move.version_group_details.add(*p_move_versions)
            p_moves.append(p_move)

        pokemon.moves.add(*p_moves)

    def __link_pokemon_form_types(self, form: PokemonForm, r_types: APIResource):
        self.__link_types(PokemonFormType, form, r_types)

    def __link_pokemon_types(self, pokemon: Pokemon):
        self.__link_types(PokemonType, pokemon, self.resource.types)

    def __link_species_pokedex_numbers(self, species: PokemonSpecies):
        r_species = self.resource.species

        for pn in r_species.pokedex_numbers:
            logger.debug("linking pokedex: %s#%d" % (str(pn.pokedex), pn.entry_number))
            species.pokedex_numbers.add(
                PokemonSpeciesDexEntry.objects.get_or_create(
                    entry_number=pn.entry_number,
                    pokedex=str(pn.pokedex),
                )[0]
            )

    def __link_species_varieties(self, species: PokemonSpecies):
        r_species = self.resource.species

        for sv in r_species.varieties:
            logger.debug("linking variety: %s" % str(sv.pokemon))
            sv_pokemon = self.__get_object_or_none(Pokemon, name=str(sv.pokemon))

            if not sv_pokemon and species.name != str(sv.pokemon):
                builder = PokemonBuilder(resource=str(sv.pokemon))
                sv_pokemon = builder.build_pokemon(link_species=True)[0]

            species.varieties.add(
                PokemonSpeciesVariety.objects.get_or_create(
                    is_default=sv.is_default,
                    pokemon=sv_pokemon,
                )[0]
            )

    def __link_species(self, pokemon: Pokemon):
        if pokemon.species:
            logger.debug("%s species already linked!" % pokemon.name)
            return

        r_species = self.resource.species

        logger.debug("linking species: %s" % r_species.name)
        species, _ = PokemonSpecies.objects.update_or_create(
            name=r_species.name,
            defaults={
                "order": r_species.order,
                "gender_rate": r_species.gender_rate,
                "capture_rate": r_species.capture_rate,
                "base_happiness": r_species.base_happiness,
                "is_baby": r_species.is_baby,
                "is_legendary": r_species.is_legendary,
                "is_mythical": r_species.is_mythical,
                "hatch_counter": r_species.hatch_counter,
                "has_gender_differences": r_species.has_gender_differences,
                "forms_switchable": r_species.forms_switchable,
                "growth_rate": r_species.growth_rate,
                "egg_groups": [g.name for g in r_species.egg_groups],
                "color": r_species.color,
                "shape": r_species.shape,
                "evolves_from_species": self.__get_evolves_from_species(),
                "generation": r_species.generation,
            },
        )

        self.__link_species_pokedex_numbers(species)
        self.__link_species_varieties(species)

        pokemon.species = species
        pokemon.save()

    def __link_stats(self, pokemon: Pokemon):
        for s in self.resource.stats:
            logger.debug("linking base %s: %d" % (s.stat, s.base_stat))
            pokemon.stats.add(
                PokemonStat.objects.get_or_create(
                    stat=s.stat,
                    effort=s.effort,
                    base_stat=s.base_stat,
                )[0]
            )

    def __link_types(
        self,
        t_model: PokemonType | PokemonFormType,
        m_obj: Pokemon | PokemonForm,
        r_types: APIResource,
    ):
        for t in r_types:
            logger.debug("linking type: %s" % t.type)
            m_obj.types.add(
                t_model.objects.get_or_create(
                    slot=t.slot,
                    type=t.type,
                )[0]
            )

    @transaction.atomic()
    def build_pokemon(self, **kwargs) -> tuple[Optional[Pokemon], bool]:
        logger.info("building Pokemon: %s..." % self.resource.name)
        pokemon, created = self.__get_pokemon()

        if kwargs.get("link_abilities", False):
            logger.debug("linking abilities...")
            self.__link_abilities(pokemon)

        if kwargs.get("link_forms", False):
            logger.debug("linking forms...")
            self.__link_forms(pokemon)

        if kwargs.get("link_game_indices", False):
            logger.debug("linking game_indices...")
            self.__link_game_indices(pokemon)

        if kwargs.get("link_moves", False):
            logger.debug("linking moves...")
            self.__link_moves(pokemon)

        if kwargs.get("link_species", False):
            logger.debug("linking species...")
            self.__link_species(pokemon)

        if kwargs.get("link_stats", False):
            logger.debug("linking stats...")
            self.__link_stats(pokemon)

        if kwargs.get("link_types", False):
            logger.debug("linking types...")
            self.__link_pokemon_types(pokemon)

        return pokemon, created

    @transaction.atomic()
    def build_pokemonform(self, **kwargs) -> tuple[Optional[PokemonForm], bool]:
        logger.info("building PokemonForm: %s..." % self.resource.name)
        return self.__get_pokemonform(self.resource)

    def __init__(self, resource: str | APIResource = "bulbasaur"):
        if isinstance(resource, APIResource):
            self.resource = resource
            return

        self.resource = pb.pokemon(resource)

In [3]:
def __sync_pokemon():
    for name in [n for n in pb.APIResourceList("pokemon").names][:1025]:
        try:
            builder = PokemonBuilder(resource=name)
            builder.build_pokemon(
                link_game_indices=True,
                link_abilities=True,
                link_species=True,
                link_forms=True,
                link_stats=True,
                link_types=True,
                # link_moves=True,
            )
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                print(f"\n[FIM] Erro HTTP 404 encontrado no ID {name}")
                break
            else:
                print(f"Erro HTTP {status_code} no ID {name}, mas continuando...")

In [4]:
def __sync_pokemonform():
    for index in [n for n in pb.APIResourceList("pokemon-form").names]:
        try:
            builder = PokemonBuilder(resource=pb.pokemon_form(index))
            builder.build_pokemonform()
        except HTTPError as e:
            status_code = getattr(
                e,
                "code",
                getattr(getattr(e, "response", None), "status_code", None),
            )

            if status_code == 404:
                print(f"\n[FIM] Erro HTTP 404 encontrado no ID {index}")
                break
            else:
                print(f"Erro HTTP {status_code} no ID {index}, mas continuando...")

In [11]:
q_str = "arceus"

forms = PokemonForm.objects.filter(
    name__istartswith=q_str,
    is_battle_only=False,
    is_default=False,
    is_mega=False,
)

(
    forms.count(),
    [f for f in forms],
)